# B4 CB 설정 비교

B1~B3와 공통 분할·평가 함수를 그대로 사용합니다. 후보와 선택 기준을 아래에서 먼저 고정합니다.
최종 평가(test·cold_final)는 수행하지 않습니다. 기존 조원 파일과 .vscode는 수정하지 않습니다.


In [1]:
from pathlib import Path
from dataclasses import asdict
from time import perf_counter
import json
import numpy as np
import pandas as pd
from bookrec.data import load_book_tags, load_split, load_to_read
from bookrec.models.content_based.tuning import ContentBasedConfig, make_content_based, rank_validation
from bookrec.models.popularity import PopularityRecommender
from bookrec.evaluation import evaluate_warm, evaluate_cold

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "results" / "content_based"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
N_WARM_USERS = 1000
configs = {}
for extended in (False, True):
    for weighting in ("binary", "count", "tfidf"):
        for profile in ("positive", "rating"):
            configs[f"C{len(configs) + 1:02d}"] = ContentBasedConfig(extended=extended, weighting=weighting, profile_weighting=profile)
# NOTE: 기본 12가지에 확장 TF-IDF의 TF 로그 완화 해제·min_df=5를 각각 두 가지 추가한다.
for profile in ("positive", "rating"):
    configs[f"C{len(configs) + 1:02d}"] = ContentBasedConfig(sublinear_tf=False, profile_weighting=profile)
for profile in ("positive", "rating"):
    configs[f"C{len(configs) + 1:02d}"] = ContentBasedConfig(min_df=5, profile_weighting=profile)
candidates = pd.DataFrame([{"candidate_id": name, **asdict(config)} for name, config in configs.items()])
candidates.to_csv(OUT / "B4_candidates.csv", index=False)
print(candidates.to_string(index=False))


candidate_id  extended weighting  min_df  sublinear_tf profile_weighting
         C01     False    binary       2          True          positive
         C02     False    binary       2          True            rating
         C03     False     count       2          True          positive
         C04     False     count       2          True            rating
         C05     False     tfidf       2          True          positive
         C06     False     tfidf       2          True            rating
         C07      True    binary       2          True          positive
         C08      True    binary       2          True            rating
         C09      True     count       2          True          positive
         C10      True     count       2          True            rating
         C11      True     tfidf       2          True          positive
         C12      True     tfidf       2          True            rating
         C13      True     tfidf       2         Fa

## 선택 기준과 평가 대상

1순위: cold_tune 이력 1·3·5·10의 NDCG@10을 동일 가중치로 평균합니다.
동점이면 warm_val NDCG@10, 그다음 설정 ID 순서로 결정합니다. 인기순은 비교 기준이며 CB 후보 선택에서 제외합니다.
기존 독자는 val의 동일한 1,000명(seed=42), 신규 독자는 cold_tune 전원 2,500명입니다.
순위 지표의 users는 정답이 있는 독자 수로, 공통 평가 함수가 지표별로 결정합니다.


In [2]:
tags = load_book_tags()
split = load_split()
to_read = load_to_read()
users = np.random.default_rng(SEED).choice(np.sort(split.val["user_id"].unique()), size=N_WARM_USERS, replace=False)
position = split.cold_tune.groupby("user_id").cumcount()
histories = [("warm_val", "train", split.train[split.train["user_id"].isin(users)])]
for size in (1, 3, 5, 10):
    histories.append(("cold_tune", str(size), split.cold_tune[position < size]))
print(f"train {len(split.train):,}건; warm 표본 {len(users):,}명; cold_tune {split.cold_tune['user_id'].nunique():,}명")


train 3,769,078건; warm 표본 1,000명; cold_tune 2,500명


## 후보별 검증 실험

각 설정은 동일한 train으로 학습합니다. 평점은 책 벡터 생성에 사용하지 않습니다.
중간 수치도 매 후보 후 CSV에 기록합니다. 실행 시간은 측정 환경에 따라 달라집니다.


In [3]:
records, diagnostics, runtimes = [], [], []
for name in ["popularity", *configs]:
    start = perf_counter()
    model = PopularityRecommender() if name == "popularity" else make_content_based(tags, configs[name])
    model.fit(split.train)
    build_fit_seconds = perf_counter() - start
    start = perf_counter()
    warm = evaluate_warm(model, split, on="val", to_read=to_read, n_users=N_WARM_USERS, seed=SEED)
    records.append({"candidate_id": name, "goal": "warm_val", "history_size": "train", **warm})
    cold = evaluate_cold(model, split, on="tune", to_read=to_read)
    for size, metrics in cold.iterrows():
        records.append({"candidate_id": name, "goal": "cold_tune", "history_size": str(size), **metrics.to_dict()})
    evaluate_seconds = perf_counter() - start
    runtimes.append({"candidate_id": name, "build_fit_seconds": build_fit_seconds, "evaluate_seconds": evaluate_seconds})
    if name != "popularity":
        for goal, size, history in histories:
            summary = model.profile_summary(history)
            diagnostics.append({"candidate_id": name, "goal": goal, "history_size": size,
                                "users": len(summary), "fallback_users": int(summary["fallback"].sum()),
                                "fallback_rate": float(summary["fallback"].mean())})
    pd.DataFrame(records).to_csv(OUT / "B4_validation_scores.csv", index=False)
    pd.DataFrame(diagnostics).to_csv(OUT / "B4_profile_summary.csv", index=False)
    pd.DataFrame(runtimes).to_csv(OUT / "B4_runtimes.csv", index=False)
    print(f"{name}: cold 평균 NDCG={cold['ndcg@10'].mean():.6f}; warm NDCG={warm['ndcg@10']:.6f}; {build_fit_seconds + evaluate_seconds:.2f}초")
    del model
scores = pd.DataFrame(records)


popularity: cold 평균 NDCG=0.221323; warm NDCG=0.041433; 1.74초
C01: cold 평균 NDCG=0.183821; warm NDCG=0.051571; 8.00초
C02: cold 평균 NDCG=0.167219; warm NDCG=0.047142; 8.50초
C03: cold 평균 NDCG=0.112029; warm NDCG=0.023210; 8.25초
C04: cold 평균 NDCG=0.095163; warm NDCG=0.022948; 8.76초
C05: cold 평균 NDCG=0.174507; warm NDCG=0.051094; 8.27초
C06: cold 평균 NDCG=0.159034; warm NDCG=0.047254; 8.87초
C07: cold 평균 NDCG=0.180887; warm NDCG=0.051281; 7.59초
C08: cold 평균 NDCG=0.164322; warm NDCG=0.046030; 8.07초
C09: cold 평균 NDCG=0.099118; warm NDCG=0.019859; 7.79초
C10: cold 평균 NDCG=0.083092; warm NDCG=0.016978; 8.29초
C11: cold 평균 NDCG=0.170602; warm NDCG=0.048862; 7.81초
C12: cold 평균 NDCG=0.154860; warm NDCG=0.045743; 8.23초
C13: cold 평균 NDCG=0.117791; warm NDCG=0.023933; 7.80초
C14: cold 평균 NDCG=0.099911; warm NDCG=0.021043; 8.24초
C15: cold 평균 NDCG=0.167980; warm NDCG=0.044470; 7.82초
C16: cold 평균 NDCG=0.151766; warm NDCG=0.041355; 8.33초


## 순위와 최종 설정

최종 설정은 위 검증 결과로만 선택합니다. 저장 파일에는 후보 설정, 선택 기준, 분할·표본 정보를 함께 기록합니다.


In [4]:
ranking = rank_validation(scores[scores["candidate_id"] != "popularity"])
ranking = ranking.merge(candidates, on="candidate_id", how="left")
ranking.to_csv(OUT / "B4_ranking.csv", index=False)
best_id = ranking.iloc[0]["candidate_id"]
baseline_id = next(name for name, config in configs.items() if config == ContentBasedConfig())
selection = {
    "candidate_id": best_id, "config": asdict(configs[best_id]),
    "criterion": "mean cold_tune NDCG@10 over history sizes 1,3,5,10; ties: warm_val NDCG@10 then candidate_id",
    "seed": SEED, "n_warm_users": N_WARM_USERS, "n_cold_users": int(split.cold_tune["user_id"].nunique()),
    "train_source": "split.train", "final_evaluation_performed": False,
    "cold_mean_ndcg@10": float(ranking.iloc[0]["cold_mean_ndcg@10"]),
    "warm_ndcg@10": float(ranking.iloc[0]["warm_ndcg@10"]),
}
(OUT / "B4_selected_config.json").write_text(json.dumps(selection, ensure_ascii=False, indent=2) + "\n")
print(ranking.to_string(index=False))
print(json.dumps(selection, ensure_ascii=False, indent=2))


candidate_id  cold_mean_ndcg@10  warm_ndcg@10  extended weighting  min_df  sublinear_tf profile_weighting
         C01           0.183821      0.051571     False    binary       2          True          positive
         C07           0.180887      0.051281      True    binary       2          True          positive
         C05           0.174507      0.051094     False     tfidf       2          True          positive
         C11           0.170602      0.048862      True     tfidf       2          True          positive
         C15           0.167980      0.044470      True     tfidf       5          True          positive
         C02           0.167219      0.047142     False    binary       2          True            rating
         C08           0.164322      0.046030      True    binary       2          True            rating
         C06           0.159034      0.047254     False     tfidf       2          True            rating
         C12           0.154860      0.045743 

## 재현과 연결 확인

선택한 JSON의 config를 읽어 같은 설정으로 모델을 재생성할 수 있는지 확인합니다.
독자·책 ID 대응, 점수 범위와 이미 읽은 책 제외를 확인합니다. 최종 평가는 하지 않습니다.


In [5]:
saved = json.loads((OUT / "B4_selected_config.json").read_text())
selected = make_content_based(tags, ContentBasedConfig(**saved["config"])).fit(split.train)
small = split.cold_tune.groupby("user_id", sort=False).head(3)
small = small[small["user_id"].isin(small["user_id"].unique()[:5])]
values = selected.score(small)
recommendations = selected.recommend(small, k=10)
assert np.isfinite(values.to_numpy()).all() and ((values.to_numpy() >= 0) & (values.to_numpy() <= 1)).all()
for user, recs in recommendations.items():
    assert len(recs) == len(set(recs)) == 10
    assert not set(recs) & set(small.loc[small["user_id"] == user, "book_id"])
print(f"설정 재생성 및 신규 독자 {len(recommendations)}명의 점수·추천 검증 통과")


설정 재생성 및 신규 독자 5명의 점수·추천 검증 통과


## 발표용 그래프

공통 setup·폰트·모델 색과 save(fig, name)를 사용합니다. 저장 경로·크기·dpi·레이아웃은 bookrec.plot_style.save()에 맡깁니다. 공용 파일은 수정하지 않습니다.


In [6]:
import os
import tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "bookrec-matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from bookrec.plot_style import setup, save, MODEL_COLORS

setup()
fig, ax = plt.subplots()
ordered = ranking.iloc[::-1]
colors = [MODEL_COLORS["cb"] if name == best_id else MODEL_COLORS["baseline"] for name in ordered["candidate_id"]]
ax.barh(ordered["candidate_id"], ordered["cold_mean_ndcg@10"], color=colors)
ax.set_xlabel("Mean cold-tune NDCG@10 (history 1, 3, 5, 10)")
ax.set_title("B4: CB configuration comparison")
save(fig, "B4_configuration_comparison")
plt.close(fig)
fig, ax = plt.subplots()
for name, label, color in [("popularity", "Popularity", MODEL_COLORS["baseline"]),
                            (baseline_id, "B3 initial CB", "#eb6834"),
                            (best_id, f"Selected CB ({best_id})", MODEL_COLORS["cb"])]:
    values = scores[(scores["candidate_id"] == name) & (scores["goal"] == "cold_tune")].copy()
    values["size"] = values["history_size"].astype(int)
    values = values.sort_values("size")
    ax.plot(values["size"], values["ndcg@10"], marker="o", color=color, label=label)
ax.set_xticks([1, 3, 5, 10])
ax.set_xlabel("History size")
ax.set_ylabel("Cold-tune NDCG@10")
ax.set_title("B4: cold-user validation")
ax.legend()
save(fig, "B4_cold_validation")
plt.close(fig)
print("발표용 그래프 2개 저장 완료 (1600 x 900)")


발표용 그래프 2개 저장 완료 (1600 x 900)
